<div align="center">
  <img src="https://raw.githubusercontent.com/NaumanHSA/neurosurfer/main/docs/assets/banner/neurosurfer-banner-light.png" alt="Neurosurfer" width="45%"/>
</div>

<br/>

# 06 — The Architect: Building Workflows from Plain English

Tutorial 03 taught you to **author** graphs by hand — nodes, edges, routers, loops. This notebook is
about the layer that authors them **for you**: the **Architect**, a ReAct agent that turns a
plain-English request into a runnable, tested, registered workflow package.

The Architect is not a template filler. It:

- **knows neurosurfer** — an auto-derived, versioned manifest of every node kind, tool, expression,
  and MCP server available *right now* (never a hand-maintained list that drifts);
- **designs and builds** the graph incrementally with a real toolbelt (`add_node`, `validate`, …),
  never registering anything invalid;
- **proves its own work** — derives acceptance criteria from your intent, actually **runs** the
  workflow, and LLM-judges the outputs;
- **reaches for control flow** (routers, loops, maps) only when the intent warrants it.

This is a **compact** walkthrough — two real builds — that exercises each of those. A "Go further"
section at the end links the heavier machinery (requirement-gathering conversation, gated
self-repair, and the A/B harness) with copy-paste snippets.

You'll learn:

1. **Self-knowledge** (`KnowledgeBase`) — what the agent knows and how it retrieves detail on demand
2. **The build loop** (`ArchitectAgent`) — watch it design → validate → register, then run the result
3. **Closed-loop verification** (`derive_acceptance` / `verify_workflow`) — the agent grading itself
4. **Conditional design** — the agent choosing a **router** for a branching intent

**Model used:** OpenAI **`gpt-5-mini`** (hosted). The Architect is the most demanding agent in
neurosurfer — long tool loops, self-judging, design revision — so this tutorial uses a strong
hosted model for reliable, reproducible builds. Any capable tool-calling model works; swap in a
local LM Studio model (as tutorial 03 does) for a free/offline run.

> **Heads-up on runtime.** Each build drives a *real* reasoning model through a multi-step tool
> loop, so the two build cells take a couple of minutes each — this is the Architect genuinely
> building and testing, not a canned demo. Hosted-model calls cost a small amount per build.

---

## Contents
1. [Setup](#1-setup)
2. [Self-Knowledge — what the Architect knows](#2-self-knowledge)
3. [The Build Loop — design, validate, register, run](#3-the-build-loop)
4. [Closed-Loop Verification — the agent grading itself](#4-closed-loop-verification)
5. [Conditional Design — the Architect picks a router](#5-conditional-design)
6. [Go Further](#6-go-further)


---

## 1. Setup

Point Python at the repo root, connect the provider, and switch on Langfuse tracing so every build
and run is inspectable.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
from pathlib import Path

repo_root = Path(os.getcwd()).parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import neurosurfer
print(f"neurosurfer {neurosurfer.__version__}")

# Quiet the runtime's per-run WARNING logs so the build/run streams read cleanly.
import logging
logging.getLogger("neurosurfer").setLevel(logging.ERROR)

In [ ]:
# ── Provider: OpenAI gpt-5-mini ─────────────────────────────────────
# Load the API key from the repo's .env (keeps it out of the notebook). To run
# locally instead, replace this cell with the LM Studio provider from tutorial 03
# (OpenAICompatProvider pointing at http://localhost:1234/v1).
for _line in (repo_root / ".env").read_text().splitlines():
    _line = _line.strip()
    if _line.startswith("OPENAI_API_KEY=") and not _line.startswith("#"):
        os.environ["OPENAI_API_KEY"] = _line.partition("=")[2].strip()

from neurosurfer.llm.providers.openai import OpenAIProvider

provider = OpenAIProvider(api_key=os.environ["OPENAI_API_KEY"], model="gpt-5-mini")
print(f"Provider: {provider.model}")
print(f"Tool call style: {provider.capabilities.tool_call_style}")

### 1.1 Tracing + a clean registry

Tracing activates automatically when `LANGFUSE_*` credentials are present before the first run.
Builds register real packages to disk, so we point the Architect at a **tutorial-local** registry
(`tutorials/.architect_tutorial/`) — nothing lands in your global store, and the notebook re-runs
cleanly.

In [ ]:
# Langfuse creds from .env (optional — everything works untraced too)
import httpx
for line in (repo_root / ".env").read_text().splitlines():
    line = line.strip()
    if line.startswith("LANGFUSE_") and "=" in line:
        k, _, v = line.partition("="); os.environ[k.strip()] = v.strip()
host = os.environ.get("LANGFUSE_HOST", "(unset)")
try:
    up = httpx.get(f"{host}/api/public/health", timeout=3.0).status_code == 200
except Exception:
    up = False
print("Tracing:", "ENABLED" if up and os.environ.get("LANGFUSE_SECRET_KEY") else "disabled")

from neurosurfer.graph.workflow.registry import WorkflowRegistry
TUT_ROOT = repo_root / "tutorials" / ".architect_tutorial"
REGISTRY_DIR, STAGING_DIR = TUT_ROOT / "registry", TUT_ROOT / "staging"
REGISTRY_DIR.mkdir(parents=True, exist_ok=True); STAGING_DIR.mkdir(parents=True, exist_ok=True)
registry = WorkflowRegistry(workflows_dir=REGISTRY_DIR)
print("Registry:", REGISTRY_DIR)

---

## 2. Self-Knowledge — what the Architect knows

Before it designs anything, the agent needs an accurate model of *what neurosurfer can do* — and
that model is **derived from live code**, never hand-written: node kinds from `_VALID_NODE_KINDS`,
fields from the pydantic schemas, expression functions from the evaluator, tools from the registry.
A freshness test fails CI if this layer ever drifts from the engine, so it can't go stale.

`KnowledgeBase` is the façade: `render_context()` (the block injected into the agent's prompt),
`describe_node_kind` / `describe_tool` / `search_docs` (on-demand retrieval, also exposed as agent
tools), and `version` (a content hash — any capability change ⇒ a new "capability set" version).

In [ ]:
from neurosurfer.architect.knowledge import KnowledgeBase

kb = KnowledgeBase()
print(f"Capability set version : {kb.version}")
print(f"neurosurfer version    : {kb.manifest['neurosurfer_version']}\n")
print("Node kinds the Architect can use:")
for kind, info in kb.manifest["node_kinds"].items():
    print(f"  {kind:9s} — {info['summary'][:84]}")

On-demand retrieval — the agent *pulls* specifics when a design decision needs them:

In [ ]:
router = kb.describe_node_kind("router")
print("router node —")
print("  summary   :", router["summary"][:96], "…")
print("  key_fields:", router["key_fields"])
print("  requires  :", router["requires"])

print("\ndocs search · 'how does a loop decide when to stop':")
for hit in kb.search_docs("how does a loop decide when to stop", k=3):
    print(f"  • {hit['heading']}  ({hit['path']})")

---

## 3. The Build Loop — design, validate, register, run

`ArchitectAgent` is a **single ReAct planner** with an 11-tool belt operating on one staged
workflow. Given an intent it sets the workflow meta, adds nodes one at a time (each validated
immediately), runs the full validation gate, and finally `register_workflow` — refusing to register
anything invalid. It never asks you a question mid-build; the intent is fixed.

The `notify` callback is the live build stream — wire it up and you can **watch it think**. We give
it a plain **linear** intent (summarise → title), and it shouldn't invent control flow it doesn't
need.

In [ ]:
from neurosurfer.architect import ArchitectAgent

def note(msg): print(f"  [architect] {msg}")

agent = ArchitectAgent(
    provider,
    registry     = registry,
    staging_root = STAGING_DIR,
    notify       = note,
    max_turns    = 30,
    verify       = "encouraged",   # test tooling available + prompted, not gated (see §4)
)

path = await agent.build(
    "Take a short article, summarise it in exactly 3 sentences, then write a catchy title "
    "for that summary."
)
print(f"\n✓ Registered package at:\n  {path}")

Inspect what it built — the nodes, kinds, and the wiring that makes it a pipeline:

In [ ]:
from neurosurfer.graph.workflow.package import load_package
from neurosurfer.graph.workflow.validate import validate_package

pkg = load_package(Path(path))
print(f"Workflow : {pkg.manifest.name}")
print(f"Inputs   : {[(i.name, i.type) for i in pkg.graph.inputs]}")
print(f"Outputs  : {pkg.graph.outputs}\n")
print("Nodes (id · kind · depends_on):")
for n in pkg.graph.nodes:
    print(f"  {n.id:16s} {n.kind:7s} depends_on={n.depends_on or '[]'}")
print(f"\nValidation: {'VALID ✓' if validate_package(pkg).ok else 'INVALID ✗'}")
assert any(n.depends_on for n in pkg.graph.nodes)   # a pipeline, not a bag of nodes
print("Wired    : yes — at least one node consumes another's output")

Now run the workflow the Architect just authored — it executes like any other package:

In [ ]:
from neurosurfer.graph.workflow import WorkflowRunner
from neurosurfer.observability.run import traced_run

ARTICLE = (
    "Researchers unveiled a solar-powered desalination rig that runs without grid electricity. "
    "In field tests it produced 20 litres of drinking water per hour from seawater using only "
    "sunlight. The team says the modular design can scale for coastal villages at a fraction of "
    "the cost of conventional plants."
)
runner = WorkflowRunner(provider)
run_inputs = {i.name: ARTICLE for i in pkg.graph.inputs}
run_inputs.setdefault("user_intent", ARTICLE)
with traced_run("tutorial:architect-run"):
    res = runner.run(pkg, run_inputs)
for key, val in res.final.items():
    print(f"[{key}]\n{str(val).strip()[:400]}\n")

---

## 4. Closed-Loop Verification — the agent grading itself

This is the machinery that turns *"quality varies"* into *"quality is measured"*. Two functions do
the work, and we drive them directly on the package from §3:

- `derive_acceptance(...)` — one LLM call → **2–6 testable criteria** + **concrete test inputs**.
- `verify_workflow(...)` — **actually runs** the workflow, then LLM-judges each criterion
  (fail-closed: an unruled criterion counts as failed); a crashed run is diagnosed deterministically.

In [ ]:
from neurosurfer.architect.agent import derive_acceptance, verify_workflow

INTENT = "Summarise an article in 3 sentences and write a catchy title for the summary."
graph_yaml = (registry.path(pkg.manifest.name) / "graph.yaml").read_text()
declared = [{"name": i.name, "type": i.type, "required": i.required} for i in pkg.graph.inputs]

plan = await derive_acceptance(provider, INTENT, graph_yaml, declared_inputs=declared)
print("Derived acceptance criteria:")
for c in plan.criteria:
    print(f"  • [{c.id}] {c.description}")
print(f"\nSynthesised test input keys: {list(plan.test_inputs)}")

In [ ]:
with traced_run("tutorial:architect-verify"):
    report = await verify_workflow(
        provider, intent=INTENT, package_dir=registry.path(pkg.manifest.name), plan=plan)

print(f"Run executed cleanly : {report.run_ok}")
print(f"Overall verdict      : {'PASSED ✓' if report.passed else 'FAILED ✗'}\n")
for v in report.verdicts:
    print(f"  {'✓' if v.get('passed') else '✗'} [{v.get('id')}] {v.get('reason','')[:100]}")
if not report.passed and report.diagnosis:
    print(f"\nDiagnosis  : {report.diagnosis[:180]}")

**Inside a build**, the `test_workflow` tool returns a FAILED report on the *error channel*, so
the model treats it as work to do — it reads the diagnosis and revises the *design*, then tests
again. Make that loop mandatory by constructing the agent with `verify="required"`:

```python
# register_workflow then REFUSES until a passing, non-stale verification exists;
# if the agent can't converge within its budget, build() raises with the last report.
agent = ArchitectAgent(provider, registry=registry, staging_root=STAGING_DIR,
                       notify=note, verify="required")
path = await agent.build("…your intent…")
```

`encouraged` (the default) is the safe choice for smaller models; reserve `required` for strong
models or strict correctness — convergence tracks model strength.

---

## 5. Conditional Design — the Architect picks a router

The agent's prompt carries a **control-flow cookbook**: different handling per category → `router`;
retry-until-good → `loop`; per-item → `map`; sometimes-applies → a `when:` guard. Crucially it's
taught *when* each is warranted — a linear task gets none (as in §3). Give it a genuinely branching
intent and it should reach for a `router` on its own.

In [ ]:
branch_agent = ArchitectAgent(
    provider, registry=registry, staging_root=STAGING_DIR, notify=note, max_turns=30,
)
bpath = await branch_agent.build(
    "Take a customer support ticket, decide whether it is urgent or routine, then draft an "
    "escalation notice for urgent tickets but a polite standard reply for routine ones."
)
bpkg = load_package(Path(bpath))

kinds = [n.kind for n in bpkg.graph.nodes]
has_router = "router" in kinds
print("\nNodes (id · kind · routes/when · depends_on):")
for n in bpkg.graph.nodes:
    ctl = f"routes={n.routes or n.cases}" if n.kind == "router" else (f"when={n.when!r}" if n.when else "")
    print(f"  {n.id:16s} {n.kind:7s} {ctl:36s} depends_on={n.depends_on or '[]'}")
print(f"\nBranching design? {'YES ✓ (router)' if has_router else 'came out linear'}")

Run one urgent ticket and confirm the router takes the escalation branch and **prunes** the
other (skipped, not errored — the control-flow semantics from tutorial 03 §10):

In [ ]:
if has_router:
    r_node = next(n for n in bpkg.graph.nodes if n.kind == "router")
    targets = list((r_node.routes or {}).values()) or [c.to for c in (r_node.cases or [])]
    ticket = "URGENT: the payment API is down and no customer can check out right now!"
    tin = {i.name: ticket for i in bpkg.graph.inputs}; tin.setdefault("user_intent", ticket)
    with traced_run("tutorial:architect-triage"):
        r = runner.run(bpkg, tin)
    taken  = [t for t in targets if t in r.nodes and not r.nodes[t].skipped]
    pruned = [t for t in targets if t in r.nodes and r.nodes[t].skipped]
    print(f"Ticket : {ticket}")
    print(f"Router : taken={taken}  pruned (skipped, not errored)={pruned}\n")
    for t in taken:
        print(f"→ {str(r.nodes[t].raw_output).strip()[:220]}")
else:
    print("No router this run — re-run the build cell (small models occasionally emit a linear design).")

---

## 6. Go Further

The same building blocks power a few heavier flows — each is one call away.

**Requirement-gathering pre-flight.** A vague ask becomes a crisp intent through a short
clarifying-question conversation before the build:

```python
from neurosurfer.architect.conversation import ArchitectConversation

async def ask(question, choices):   # CLI renders an arrow-key menu; return the answer
    ...
intent, answers = await ArchitectConversation(provider).run(
    "read incoming support emails and sort them", ask=ask)
path = await ArchitectAgent(provider, registry=registry).build(intent, answers=answers)
```

**Self-repairing (gated) builds.** `verify="required"` runs the full design → test → diagnose →
revise loop and won't register until it passes (§4).

**A/B harness — ReAct vs. the legacy pipeline.** The ReAct agent replaced a fixed staged pipeline;
the decision to retire that baseline is *evidence-driven*. Measure them side by side:

```python
from neurosurfer.architect.agent import run_harness, render_report, default_builders, SUITE_BASIC

builders = default_builders(provider, registry=registry, staging_root=STAGING_DIR)
results  = await run_harness(SUITE_BASIC, builders)   # validity, node count, latency per builder
print(render_report(results))
```

---

## Summary

| Capability | API | What it gives you |
|---|---|---|
| **Self-knowledge** | `KnowledgeBase().render_context()` · `.describe_node_kind()` · `.search_docs()` | An always-current, versioned model of every node kind/tool/expression — derived from code, tested against drift |
| **Autonomous build** | `await ArchitectAgent(provider).build(intent)` | Design → per-node validation → full gate → registration, in one ReAct session; never registers invalid |
| **Watch it build** | `ArchitectAgent(..., notify=cb)` | A live stream of every build step |
| **Acceptance criteria** | `derive_acceptance(provider, intent, graph_yaml, declared_inputs=)` | Testable criteria + synthesised inputs from the intent |
| **Self-verification** | `verify_workflow(provider, intent=, package_dir=, plan=)` | Really runs the workflow, LLM-judges each criterion, diagnoses failures |
| **Gated self-repair** | `ArchitectAgent(..., verify="required")` | Registration blocked until a passing, non-stale test |
| **Conditional design** | (automatic) | Reaches for `router` / `loop` / `map` only when the intent warrants it |
| **A/B evaluation** | `run_harness(intents, default_builders(provider))` | ReAct agent vs. legacy pipeline, measured |

**Key ideas**
- The Architect **knows neurosurfer from code**, not a hand-maintained list — designs stay valid as
  the framework evolves, and a freshness test proves it can't drift.
- A build is a real **ReAct loop with a toolbelt** on one staged package — it iterates cheaply and
  **never registers an invalid workflow**.
- Quality is **measured, not hoped for**: derive criteria → run → judge → (in `required` mode) don't
  ship until they pass.
- Control flow is **earned**: a linear intent yields a linear graph; a branching intent yields a
  router.

Open **http://localhost:3000 → Traces** to explore every `tutorial:*` and `ArchitectAgent.build`
run — the design loop, each tool call, the judge, and token costs.
